# SQL aplicado a la limpieza de datos de mantenimiento predictivo

## Caso real `MachineData.zip` con DuckDB en Google Colab

**Duración sugerida:** 2 horas  
**Motor:** DuckDB SQL  
**Infraestructura:** únicamente Google Colab; no requiere GCP, BigQuery, cuenta de facturación ni credenciales.

Este notebook trabaja directamente con los **63 CSV reales** del ZIP. Los archivos contienen nueve subsistemas de una máquina industrial:

| Tabla | Registros | Contenido |
|---|---:|---|
| `Drilling` | 183 | Empuje, torque, velocidad y penetración |
| `Hydraulics` | 1,213 | Presiones y eficiencias hidráulicas |
| `Inputs` | 1,337 | Registros de entradas digitales |
| `Lubrication` | 85 | Presión y flujo de lubricación |
| `Motors` | 508 | Corriente, voltaje, temperatura y potencia |
| `Outputs` | 31,481 | Registros de salidas digitales |
| `Services` | 73 | Tiempo restante para servicios |
| `Setup` | 1,065 | Parámetros de configuración |
| `Water` | 74 | Flujo y presión de agua |

### Objetivos

1. Leer múltiples CSV directamente con DuckDB.
2. Crear capas `raw`, `clean` y `features`.
3. Detectar nulos, valores inválidos, ceros y timestamps repetidos.
4. Aplicar `TRY_CAST`, `NULLIF`, `COALESCE`, `CASE` y funciones de texto.
5. Usar `ROW_NUMBER`, `LAG`, ventanas móviles y `time_bucket`.
6. Integrar series con frecuencias diferentes sin multiplicar registros.
7. Crear features interpretables sin inventar etiquetas de falla.

> El ZIP no contiene eventos de falla ni órdenes de mantenimiento. El resultado sirve para análisis, preparación de features y detección de anomalías, pero no demuestra que una máquina vaya a fallar.

## Arquitectura

```text
MachineData.zip
      │
      ▼
DuckDB read_csv()
      │
      ▼
9 tablas raw
      │
      ▼
9 tablas clean
      │
      ▼
Agregación por minuto
      │
      ▼
features_machine_1min
      │
      ▼
machine_data.duckdb
```

Python se usa únicamente para subir/descomprimir el ZIP y conectar el motor. Toda la transformación de datos se realiza con SQL.

## 0. Preparación del entorno

Ejecuta las celdas en orden. La instalación tarda unos segundos.

In [4]:
!pip -q install --upgrade duckdb

In [5]:
import duckdb
import zipfile
import tempfile
from pathlib import Path
from IPython.display import display

print("DuckDB:", duckdb.__version__)

DuckDB: 1.5.5


## 1. Subir y extraer `MachineData.zip`

Selecciona el ZIP original cuando aparezca el cuadro de carga.

In [10]:
from google.colab import files

uploaded = files.upload()
zip_candidates = [name for name in uploaded if name.lower().endswith(".zip")]

if not zip_candidates:
    raise FileNotFoundError("Debes subir MachineData.zip")

zip_name = zip_candidates[0]
extract_dir = Path(tempfile.mkdtemp(prefix="machinedata_"))

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(extract_dir)

machine_roots = list(extract_dir.rglob("MachineData"))
if not machine_roots:
    raise FileNotFoundError("No se encontró la carpeta MachineData dentro del ZIP")

MACHINE_ROOT = machine_roots[0]
csv_files = sorted(MACHINE_ROOT.rglob("*.csv"))

print("Carpeta:", MACHINE_ROOT)
print("CSV encontrados:", len(csv_files))

if len(csv_files) != 63:
    print("⚠️ El ZIP analizado originalmente contenía 63 CSV.")

Saving MachineData.zip to MachineData (1).zip
Carpeta: /tmp/machinedata_th9aio6g/MachineData
CSV encontrados: 63


## 2. Crear la base DuckDB

La base queda almacenada en `/content/machine_data.duckdb`. Al final podrás descargarla y volverla a abrir sin reconstruir las tablas.

In [9]:
DB_PATH = "/content/machine_data.duckdb"
con = duckdb.connect(DB_PATH)

def show_sql(query, max_rows=20):
    """Ejecuta una consulta y muestra un DataFrame."""
    df = con.sql(query).df()
    display(df.head(max_rows))
    print(f"Filas devueltas: {len(df):,}")
    return df

def run_sql(query):
    """Ejecuta DDL/DML."""
    con.execute(query)
    print("✅ SQL ejecutado")

print("Base creada:", DB_PATH)

Base creada: /content/machine_data.duckdb


## 3. Crear las nueve tablas `raw`

Los CSV tienen:

- Primera línea: `sep=|`
- Delimitador real: `|`
- Segunda línea: encabezado

`read_csv()` leerá todos los archivos de cada subsistema mediante un patrón. `all_varchar = true` conserva la capa original como texto, permitiendo practicar la conversión y auditar valores inválidos.

In [11]:
SUBSYSTEMS = [
    "Drilling", "Hydraulics", "Inputs", "Lubrication", "Motors",
    "Outputs", "Services", "Setup", "Water"
]

for subsystem in SUBSYSTEMS:
    table_name = f"raw_{subsystem.lower()}"
    file_pattern = str(MACHINE_ROOT / "*" / "*" / "*" / subsystem / "*.csv")

    sql = f"""
    CREATE OR REPLACE TABLE {table_name} AS
    SELECT
        row_number() OVER () AS ingestion_row,
        * EXCLUDE (filename),
        filename AS source_file,
        regexp_extract(filename, '[0-9]{{4}}-[0-9]{{2}}-[0-9]{{2}}', 0)
            AS source_date
    FROM read_csv(
        '{file_pattern}',
        delim = '|',
        skip = 1,
        header = true,
        all_varchar = true,
        union_by_name = true,
        filename = true,
        strict_mode = false,
        null_padding = true
    );
    """
    con.execute(sql)

print("✅ Tablas raw creadas")

✅ Tablas raw creadas


In [12]:
show_sql("""
SELECT
    table_name
FROM information_schema.tables
WHERE table_schema = 'main'
  AND table_name LIKE 'raw_%'
ORDER BY table_name
""")

,table_name
0,raw_drilling
1,raw_hydraulics
2,raw_inputs
3,raw_lubrication
4,raw_motors
5,raw_outputs
6,raw_services
7,raw_setup
8,raw_water


Filas devueltas: 9


,table_name
0,raw_drilling
1,raw_hydraulics
2,raw_inputs
3,raw_lubrication
4,raw_motors
5,raw_outputs
6,raw_services
7,raw_setup
8,raw_water


### Ejercicio 1 — Inventario y reconciliación

Combina los conteos de las nueve tablas con `UNION ALL`. Deben sumar **36,019 registros**.

In [14]:
counts = show_sql("""
SELECT 'raw_drilling' AS table_name, COUNT(*) AS rows FROM raw_drilling
UNION ALL SELECT 'raw_hydraulics', COUNT(*) FROM raw_hydraulics
UNION ALL SELECT 'raw_inputs', COUNT(*) FROM raw_inputs
UNION ALL SELECT 'raw_lubrication', COUNT(*) FROM raw_lubrication
UNION ALL SELECT 'raw_motors', COUNT(*) FROM raw_motors
UNION ALL SELECT 'raw_outputs', COUNT(*) FROM raw_outputs
UNION ALL SELECT 'raw_services', COUNT(*) FROM raw_services
UNION ALL SELECT 'raw_setup', COUNT(*) FROM raw_setup
UNION ALL SELECT 'raw_water', COUNT(*) FROM raw_water
ORDER BY table_name
""")

assert counts["rows"].sum() == 36019, "El total no coincide con el ZIP analizado"
print("✅ Total validado:", counts["rows"].sum())

,table_name,rows
0,raw_drilling,183
1,raw_hydraulics,1213
2,raw_inputs,1337
3,raw_lubrication,85
4,raw_motors,508
5,raw_outputs,31481
6,raw_services,73
7,raw_setup,1065
8,raw_water,74


Filas devueltas: 9
✅ Total validado: 36019


## 4. Diagnóstico inicial

Antes de limpiar debemos determinar:

- Si fecha y hora son convertibles.
- Qué columnas contienen nulos.
- Qué sensores permanecen en cero.
- Si existen varias observaciones en el mismo segundo.
- Qué modos operativos están presentes.

In [15]:
show_sql("""
SELECT
    COUNT(*) AS total_rows,
    count(*) FILTER (
        WHERE try_strptime(Date || ' ' || Time, '%d/%m/%Y %H:%M:%S') IS NULL
    ) AS invalid_timestamp,
    count(*) FILTER (
        WHERE TRY_CAST(ThrustToHead AS DOUBLE) IS NULL
    ) AS invalid_thrust,
    count(*) FILTER (
        WHERE TRY_CAST(Torque AS DOUBLE) IS NULL
    ) AS invalid_torque,
    count(*) FILTER (
        WHERE TRY_CAST(Penetration AS DOUBLE) = 0
    ) AS zero_penetration,
    COUNT(DISTINCT Date || ' ' || Time) AS unique_timestamps
FROM raw_drilling
""")

,total_rows,invalid_timestamp,invalid_thrust,invalid_torque,zero_penetration,unique_timestamps
0,183,0,0,0,183,180


Filas devueltas: 1


,total_rows,invalid_timestamp,invalid_thrust,invalid_torque,zero_penetration,unique_timestamps
0,183,0,0,0,183,180


In [16]:
show_sql("""
SELECT
    Status_Process,
    Status_Mode,
    COUNT(*) AS observations
FROM raw_drilling
GROUP BY Status_Process, Status_Mode
ORDER BY observations DESC
""")

,Status_Process,Status_Mode,observations
0,Piloting Mode Selected,RF,51
1,No Operating Mode Selected,Not Selected,46
2,Pulling Mode Selected,MU,13
3,Pulling Mode Selected,BO,12
4,No Operating Mode Selected,RF,10
5,Piloting Mode Selected,RR,10
6,Reaming Mode Selected,MU,8
7,Reaming Mode Selected,RF,7
8,Reaming Mode Selected,BO,7
9,Piloting Mode Selected,MU,5


Filas devueltas: 16


,Status_Process,Status_Mode,observations
0,Piloting Mode Selected,RF,51
1,No Operating Mode Selected,Not Selected,46
2,Pulling Mode Selected,MU,13
3,Pulling Mode Selected,BO,12
4,No Operating Mode Selected,RF,10
5,Piloting Mode Selected,RR,10
6,Reaming Mode Selected,MU,8
7,Reaming Mode Selected,RF,7
8,Reaming Mode Selected,BO,7
9,Piloting Mode Selected,MU,5


### Ejercicio 2 — Perfil de motores

Calcula:

1. Total de registros.
2. Porcentaje de `Motor1Temp` igual a cero.
3. Porcentaje de `Drive2Current` igual a cero.
4. Mínimo y máximo de `Drive1Current`.

Usa `TRY_CAST`, `FILTER`, `COUNT`, `MIN` y `MAX`.

In [17]:
# Solución
show_sql("""
SELECT
    COUNT(*) AS total_rows,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE TRY_CAST(Motor1Temp AS DOUBLE) = 0)
        / NULLIF(COUNT(*), 0),
        2
    ) AS motor1_temp_zero_pct,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE TRY_CAST(Drive2Current AS DOUBLE) = 0)
        / NULLIF(COUNT(*), 0),
        2
    ) AS drive2_current_zero_pct,
    MIN(TRY_CAST(Drive1Current AS DOUBLE)) AS drive1_current_min,
    MAX(TRY_CAST(Drive1Current AS DOUBLE)) AS drive1_current_max
FROM raw_motors
""")

,total_rows,motor1_temp_zero_pct,drive2_current_zero_pct,drive1_current_min,drive1_current_max
0,508,99.41,100.0,-59.0,519.0


Filas devueltas: 1


,total_rows,motor1_temp_zero_pct,drive2_current_zero_pct,drive1_current_min,drive1_current_max
0,508,99.41,100.0,-59.0,519.0


### Interpretación

- `Motor1Temp` está en cero aproximadamente 99.41%.
- `Drive2Current` está en cero el 100%.
- Cero puede significar paro, sensor deshabilitado, falta de comunicación o valor físico real.
- No se deben sustituir automáticamente todos los ceros por la media.

## 5. Limpieza de las tablas analógicas

Aplicaremos:

- `try_strptime()` para fecha y hora.
- `TRY_CAST()` para números.
- `NULLIF(TRIM(...), '')` para textos vacíos.
- Nombres en `snake_case`.
- `md5()` para crear un identificador auditable.

`TRY_CAST('error' AS DOUBLE)` devuelve `NULL`; `CAST('error' AS DOUBLE)` detendría la consulta.

In [18]:
run_sql("""
CREATE OR REPLACE TABLE clean_drilling AS
SELECT
    md5(source_file || '#' || ingestion_row::VARCHAR) AS record_id,
    try_strptime(Date || ' ' || Time, '%d/%m/%Y %H:%M:%S') AS timestamp,
    NULLIF(TRIM(Status_Process), '') AS status_process,
    NULLIF(TRIM(Status_Mode), '') AS status_mode,
    TRY_CAST(ThrustToHead AS DOUBLE) AS thrust_to_head,
    TRY_CAST(ThrustPerCutter AS DOUBLE) AS thrust_per_cutter,
    TRY_CAST(ChuckSpeed AS DOUBLE) AS chuck_speed,
    TRY_CAST(Torque AS DOUBLE) AS torque,
    TRY_CAST(Penetration AS DOUBLE) AS penetration,
    TRY_CAST(Ana_Distance AS DOUBLE) AS ana_distance,
    TRY_CAST(RemoteControlEnable AS BOOLEAN) AS remote_control_enable,
    TRY_CAST(source_date AS DATE) AS source_date,
    source_file,
    ingestion_row
FROM raw_drilling;
""")

run_sql("""
CREATE OR REPLACE TABLE clean_hydraulics AS
SELECT
    md5(source_file || '#' || ingestion_row::VARCHAR) AS record_id,
    try_strptime(Date || ' ' || Time, '%d/%m/%Y %H:%M:%S') AS timestamp,
    NULLIF(TRIM(Status_Process), '') AS status_process,
    NULLIF(TRIM(Status_Mode), '') AS status_mode,
    TRY_CAST(PressureA AS DOUBLE) AS pressure_a,
    TRY_CAST(PressureB AS DOUBLE) AS pressure_b,
    TRY_CAST(OilTemp AS DOUBLE) AS oil_temperature,
    TRY_CAST(PumpPress AS DOUBLE) AS pump_pressure,
    TRY_CAST(HydMotorTemp AS DOUBLE) AS hydraulic_motor_temperature,
    TRY_CAST(HydFlowEfficiency AS DOUBLE) AS hydraulic_flow_efficiency,
    TRY_CAST(HydPressEfficiency AS DOUBLE) AS hydraulic_pressure_efficiency,
    TRY_CAST(HydSystemEfficiencyn AS DOUBLE) AS hydraulic_system_efficiency,
    TRY_CAST(source_date AS DATE) AS source_date,
    source_file,
    ingestion_row
FROM raw_hydraulics;
""")

✅ SQL ejecutado
✅ SQL ejecutado


In [19]:
run_sql("""
CREATE OR REPLACE TABLE clean_motors AS
SELECT
    md5(source_file || '#' || ingestion_row::VARCHAR) AS record_id,
    try_strptime(Date || ' ' || Time, '%d/%m/%Y %H:%M:%S') AS timestamp,
    NULLIF(TRIM(Status_Process), '') AS status_process,
    NULLIF(TRIM(Status_Mode), '') AS status_mode,
    TRY_CAST(Motor1Temp AS DOUBLE) AS motor_1_temperature,
    TRY_CAST(Drive1Current AS DOUBLE) AS drive_1_current,
    TRY_CAST(Motor2Temp AS DOUBLE) AS motor_2_temperature,
    TRY_CAST(Drive2Current AS DOUBLE) AS drive_2_current,
    TRY_CAST(GBTemp AS DOUBLE) AS gearbox_temperature,
    TRY_CAST(Voltage AS DOUBLE) AS voltage,
    TRY_CAST(Drive1Voltage AS DOUBLE) AS drive_1_voltage,
    TRY_CAST(Drive2Voltage AS DOUBLE) AS drive_2_voltage,
    TRY_CAST(TotalDrivePower AS DOUBLE) AS total_drive_power,
    TRY_CAST(KVA AS BOOLEAN) AS kva_flag,
    TRY_CAST(source_date AS DATE) AS source_date,
    source_file,
    ingestion_row
FROM raw_motors;
""")

run_sql("""
CREATE OR REPLACE TABLE clean_lubrication AS
SELECT
    md5(source_file || '#' || ingestion_row::VARCHAR) AS record_id,
    try_strptime(Date || ' ' || Time, '%d/%m/%Y %H:%M:%S') AS timestamp,
    NULLIF(TRIM(Status_Process), '') AS status_process,
    NULLIF(TRIM(Status_Mode), '') AS status_mode,
    TRY_CAST(LubePress AS DOUBLE) AS lubrication_pressure,
    TRY_CAST(LubeFlow AS DOUBLE) AS lubrication_flow,
    source_file,
    ingestion_row
FROM raw_lubrication;
""")

run_sql("""
CREATE OR REPLACE TABLE clean_water AS
SELECT
    md5(source_file || '#' || ingestion_row::VARCHAR) AS record_id,
    try_strptime(Date || ' ' || Time, '%d/%m/%Y %H:%M:%S') AS timestamp,
    NULLIF(TRIM(Status_Process), '') AS status_process,
    NULLIF(TRIM(Status_Mode), '') AS status_mode,
    TRY_CAST(WaterFlow AS DOUBLE) AS water_flow,
    TRY_CAST(WaterPressure AS DOUBLE) AS water_pressure,
    source_file,
    ingestion_row
FROM raw_water;
""")

✅ SQL ejecutado
✅ SQL ejecutado
✅ SQL ejecutado


## 6. Limpieza de tablas de soporte

Las entradas y salidas digitales se conservan como enteros. Más adelante se demostrarán como palabras de bits del PLC.

In [20]:
run_sql("""
CREATE OR REPLACE TABLE clean_inputs AS
SELECT
    md5(source_file || '#' || ingestion_row::VARCHAR) AS record_id,
    try_strptime(Date || ' ' || Time, '%d/%m/%Y %H:%M:%S') AS timestamp,
    NULLIF(TRIM(Status_Process), '') AS status_process,
    NULLIF(TRIM(Status_Mode), '') AS status_mode,
    TRY_CAST(Status_DigInputs0 AS BIGINT) AS digital_inputs_0,
    TRY_CAST(Status_DigInputs1 AS BIGINT) AS digital_inputs_1,
    TRY_CAST(Status_DigInputs2 AS BIGINT) AS digital_inputs_2,
    TRY_CAST(Status_DigInputs3 AS BIGINT) AS digital_inputs_3,
    TRY_CAST(Status_DigInputs4 AS BIGINT) AS digital_inputs_4,
    TRY_CAST(Status_DigInputs5 AS BIGINT) AS digital_inputs_5,
    TRY_CAST(Status_DigInputs6 AS BIGINT) AS digital_inputs_6,
    source_file,
    ingestion_row
FROM raw_inputs;

CREATE OR REPLACE TABLE clean_outputs AS
SELECT
    md5(source_file || '#' || ingestion_row::VARCHAR) AS record_id,
    try_strptime(Date || ' ' || Time, '%d/%m/%Y %H:%M:%S') AS timestamp,
    NULLIF(TRIM(Status_Process), '') AS status_process,
    NULLIF(TRIM(Status_Mode), '') AS status_mode,
    TRY_CAST(Status_DigOutputs0 AS BIGINT) AS digital_outputs_0,
    TRY_CAST(Status_DigOutputs1 AS BIGINT) AS digital_outputs_1,
    TRY_CAST(Status_DigOutputs2 AS BIGINT) AS digital_outputs_2,
    TRY_CAST(Status_DigOutputs3 AS BIGINT) AS digital_outputs_3,
    source_file,
    ingestion_row
FROM raw_outputs;
""")

✅ SQL ejecutado


In [21]:
run_sql("""
CREATE OR REPLACE TABLE clean_services AS
SELECT
    md5(source_file || '#' || ingestion_row::VARCHAR) AS record_id,
    try_strptime(Date || ' ' || Time, '%d/%m/%Y %H:%M:%S') AS timestamp,
    NULLIF(TRIM(Status_Process), '') AS status_process,
    NULLIF(TRIM(Status_Mode), '') AS status_mode,
    TRY_CAST(Status_MachineServiceTimeRem AS DOUBLE) AS machine_service_time_remaining,
    TRY_CAST(Status_ChuckBoltsServiceTimeRem AS DOUBLE) AS chuck_bolts_service_time_remaining,
    TRY_CAST(Status_ChuckServiceTimeRem AS DOUBLE) AS chuck_service_time_remaining,
    TRY_CAST(Status_GB1FirstServiceTimeRem AS DOUBLE) AS gb1_first_service_time_remaining,
    TRY_CAST(Status_GB1SecondServiceTimeRem AS DOUBLE) AS gb1_second_service_time_remaining,
    TRY_CAST(Status_GB2FirstServiceTimeRem AS DOUBLE) AS gb2_first_service_time_remaining,
    TRY_CAST(Status_GB2SecondServiceTimeRem AS DOUBLE) AS gb2_second_service_time_remaining,
    TRY_CAST(Status_ElecPackServiceTimeRem AS DOUBLE) AS electric_pack_service_time_remaining,
    TRY_CAST(Status_HydPackServiceTimeRem AS DOUBLE) AS hydraulic_pack_service_time_remaining,
    TRY_CAST(Status_DCMotorServiceTimeRem AS DOUBLE) AS dc_motor_service_time_remaining,
    TRY_CAST(Status_LubeServiceTimeRem AS DOUBLE) AS lubrication_service_time_remaining,
    source_file,
    ingestion_row
FROM raw_services;

CREATE OR REPLACE TABLE clean_setup AS
SELECT
    md5(source_file || '#' || ingestion_row::VARCHAR) AS record_id,
    try_strptime(Date || ' ' || Time, '%d/%m/%Y %H:%M:%S') AS timestamp,
    NULLIF(TRIM(Status_Process), '') AS status_process,
    NULLIF(TRIM(Status_Mode), '') AS status_mode,
    TRY_CAST(Status_SelectedTorqueKN AS DOUBLE) AS selected_torque_kn,
    TRY_CAST(Status_CylBoreSize AS DOUBLE) AS cylinder_bore_size,
    TRY_CAST(Status_CylRodSize AS DOUBLE) AS cylinder_rod_size,
    TRY_CAST(Status_CylQty AS INTEGER) AS cylinder_quantity,
    TRY_CAST(Status_LowGearNum AS DOUBLE) AS low_gear_number,
    TRY_CAST(Status_ServicePassword AS DOUBLE) AS service_password,
    TRY_CAST(CutterQtyOp AS DOUBLE) AS cutter_quantity_operation,
    TRY_CAST(CurrentWeight AS DOUBLE) AS current_weight,
    TRY_CAST(SYPTVersionLog AS DOUBLE) AS sypt_version_log,
    TRY_CAST(IWSVersionLog AS DOUBLE) AS iws_version_log,
    source_file,
    ingestion_row
FROM raw_setup;
""")

✅ SQL ejecutado


## 7. Duplicados temporales

Existen varias observaciones dentro del mismo segundo, especialmente en `Outputs`. No son filas exactamente iguales.

La regla siguiente conserva una observación por segundo para motores. `ROW_NUMBER()` hace que la decisión sea reproducible.

In [22]:
show_sql("""
SELECT
    timestamp,
    COUNT(*) AS observations_in_second
FROM clean_motors
GROUP BY timestamp
HAVING COUNT(*) > 1
ORDER BY observations_in_second DESC, timestamp
LIMIT 20
""")

,timestamp,observations_in_second
0,2025-12-15 12:07:35,4
1,2025-12-11 10:33:37,3
2,2025-12-11 10:35:37,3
3,2025-12-11 10:35:54,3
4,2025-12-11 10:35:57,3
5,2025-12-11 10:36:48,3
6,2025-12-11 10:36:50,3
7,2025-12-11 10:36:53,3
8,2025-12-11 10:37:08,3
9,2025-12-11 10:43:24,3


Filas devueltas: 20


,timestamp,observations_in_second
0,2025-12-15 12:07:35,4
1,2025-12-11 10:33:37,3
2,2025-12-11 10:35:37,3
3,2025-12-11 10:35:54,3
4,2025-12-11 10:35:57,3
5,2025-12-11 10:36:48,3
6,2025-12-11 10:36:50,3
7,2025-12-11 10:36:53,3
8,2025-12-11 10:37:08,3
9,2025-12-11 10:43:24,3


In [23]:
run_sql("""
CREATE OR REPLACE VIEW dedup_motors AS
SELECT * EXCLUDE (row_priority)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY timestamp
            ORDER BY source_file DESC, ingestion_row DESC
        ) AS row_priority
    FROM clean_motors
)
WHERE row_priority = 1;
""")

✅ SQL ejecutado


### Ejercicio 3 — Duplicados hidráulicos

Crea `dedup_hydraulics` utilizando `ROW_NUMBER()` y conserva la última observación de cada timestamp.

In [24]:
# Solución
run_sql("""
CREATE OR REPLACE VIEW dedup_hydraulics AS
SELECT * EXCLUDE (row_priority)
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY timestamp
            ORDER BY source_file DESC, ingestion_row DESC
        ) AS row_priority
    FROM clean_hydraulics
)
WHERE row_priority = 1;
""")

✅ SQL ejecutado


## 8. Registros digitales del PLC

Valores como `1023`, `16384`, `28688` o `-32768` indican que las columnas digitales probablemente contienen palabras de bits, no mediciones continuas.

La siguiente consulta decodifica los 16 bits de `digital_outputs_2`. Para asignar nombres físicos se necesita el mapa del PLC.

In [25]:
show_sql("""
WITH decoded AS (
    SELECT
        timestamp,
        digital_outputs_2 AS register_value,
        bit_number,
        CASE
            WHEN (digital_outputs_2 & (1::BIGINT << bit_number)) <> 0
            THEN 1 ELSE 0
        END AS bit_state
    FROM clean_outputs
    CROSS JOIN generate_series(0, 15) AS bits(bit_number)
)
SELECT
    bit_number,
    COUNT(*) FILTER (WHERE bit_state = 1) AS active_observations,
    ROUND(100.0 * AVG(bit_state), 2) AS active_pct
FROM decoded
GROUP BY bit_number
ORDER BY bit_number
""")

,bit_number,active_observations,active_pct
0,0,9042,28.72
1,1,0,0.00
2,2,18187,57.77
3,3,0,0.00
4,4,29527,93.79
5,5,0,0.00
6,6,4749,15.09
7,7,1609,5.11
8,8,6960,22.11
9,9,3977,12.63


Filas devueltas: 16


,bit_number,active_observations,active_pct
0,0,9042,28.72
1,1,0,0.00
2,2,18187,57.77
3,3,0,0.00
4,4,29527,93.79
5,5,0,0.00
6,6,4749,15.09
7,7,1609,5.11
8,8,6960,22.11
9,9,3977,12.63


> No llames a un bit “alarma”, “bomba” o “falla” sin documentación técnica. La consulta identifica actividad binaria, no su significado.

## 9. Sincronización por ventanas de un minuto

Las frecuencias reales son diferentes:

| Subsistema | Intervalo mediano aproximado |
|---|---:|
| Motores | 2 s |
| Hidráulica | 2 s |
| Perforación | 22 s |
| Lubricación | 605 s |
| Agua | 1,033 s |

Un `JOIN` exacto por timestamp perdería datos o produciría combinaciones muchos-a-muchos. Primero agregaremos cada tabla con `time_bucket()`.

In [26]:
run_sql("""
CREATE OR REPLACE TABLE agg_motors_1min AS
SELECT
    time_bucket(INTERVAL '1 minute', timestamp) AS bucket_1min,
    AVG(NULLIF(drive_1_current, 0)) AS drive_1_current_avg_nonzero,
    MAX(ABS(drive_1_current)) AS drive_1_current_abs_max,
    AVG(NULLIF(voltage, 0)) AS voltage_avg_nonzero,
    AVG(NULLIF(total_drive_power, 0)) AS total_drive_power_avg_nonzero,
    MAX(ABS(total_drive_power)) AS total_drive_power_abs_max,
    COUNT(*) AS motor_samples,
    COUNT(*) FILTER (WHERE drive_1_current = 0) AS motor_zero_current_samples
FROM clean_motors
GROUP BY bucket_1min;

CREATE OR REPLACE TABLE agg_hydraulics_1min AS
SELECT
    time_bucket(INTERVAL '1 minute', timestamp) AS bucket_1min,
    AVG(NULLIF(pressure_a, 0)) AS pressure_a_avg_nonzero,
    MAX(pressure_a) AS pressure_a_max,
    AVG(NULLIF(pressure_b, 0)) AS pressure_b_avg_nonzero,
    AVG(NULLIF(pump_pressure, 0)) AS pump_pressure_avg_nonzero,
    MAX(pump_pressure) AS pump_pressure_max,
    AVG(NULLIF(hydraulic_pressure_efficiency, 0))
        AS hydraulic_pressure_efficiency_avg_nonzero,
    AVG(NULLIF(hydraulic_system_efficiency, 0))
        AS hydraulic_system_efficiency_avg_nonzero,
    COUNT(*) AS hydraulic_samples
FROM clean_hydraulics
GROUP BY bucket_1min;
""")

✅ SQL ejecutado


In [27]:
run_sql("""
CREATE OR REPLACE TABLE agg_drilling_1min AS
SELECT
    time_bucket(INTERVAL '1 minute', timestamp) AS bucket_1min,
    arg_max(status_process, timestamp) AS status_process,
    arg_max(status_mode, timestamp) AS status_mode,
    AVG(NULLIF(thrust_to_head, 0)) AS thrust_to_head_avg_nonzero,
    MAX(ABS(thrust_to_head)) AS thrust_to_head_abs_max,
    AVG(NULLIF(chuck_speed, 0)) AS chuck_speed_avg_nonzero,
    MAX(chuck_speed) AS chuck_speed_max,
    AVG(NULLIF(torque, 0)) AS torque_avg_nonzero,
    MAX(ABS(torque)) AS torque_abs_max,
    AVG(ana_distance) AS ana_distance_avg,
    COUNT(*) AS drilling_samples
FROM clean_drilling
GROUP BY bucket_1min;
""")

✅ SQL ejecutado


### Ejercicio 4 — Efecto de los ceros

Compara la corriente promedio:

1. Incluyendo ceros.
2. Excluyendo ceros con `NULLIF`.
3. Solo durante un estado de operación.

In [28]:
# Solución
show_sql("""
SELECT
    AVG(drive_1_current) AS avg_including_zeros,
    AVG(NULLIF(drive_1_current, 0)) AS avg_excluding_zeros,
    AVG(
        CASE
            WHEN status_process <> 'No Operating Mode Selected'
            THEN NULLIF(drive_1_current, 0)
        END
    ) AS avg_when_operating
FROM clean_motors
""")

,avg_including_zeros,avg_excluding_zeros,avg_when_operating
0,1.55315,2.286957,2.286957


Filas devueltas: 1


,avg_including_zeros,avg_excluding_zeros,avg_when_operating
0,1.55315,2.286957,2.286957


`NULLIF(valor, 0)` no significa que todos los ceros sean errores. Permite comparar una métrica global contra la señal cuando el sensor reporta un valor distinto de cero.

## 10. Integración de perforación, motores e hidráulica

Después de agregar a un minuto, usamos `FULL OUTER JOIN` para conservar ventanas presentes en cualquiera de los tres subsistemas.

In [29]:
run_sql("""
CREATE OR REPLACE TABLE machine_1min_base AS
SELECT
    COALESCE(d.bucket_1min, m.bucket_1min, h.bucket_1min) AS bucket_1min,
    d.status_process,
    d.status_mode,
    d.thrust_to_head_avg_nonzero,
    d.thrust_to_head_abs_max,
    d.chuck_speed_avg_nonzero,
    d.chuck_speed_max,
    d.torque_avg_nonzero,
    d.torque_abs_max,
    d.ana_distance_avg,
    m.drive_1_current_avg_nonzero,
    m.drive_1_current_abs_max,
    m.voltage_avg_nonzero,
    m.total_drive_power_avg_nonzero,
    m.total_drive_power_abs_max,
    h.pressure_a_avg_nonzero,
    h.pressure_a_max,
    h.pressure_b_avg_nonzero,
    h.pump_pressure_avg_nonzero,
    h.pump_pressure_max,
    h.hydraulic_pressure_efficiency_avg_nonzero,
    h.hydraulic_system_efficiency_avg_nonzero,
    d.drilling_samples,
    m.motor_samples,
    h.hydraulic_samples
FROM agg_drilling_1min d
FULL OUTER JOIN agg_motors_1min m
    ON d.bucket_1min = m.bucket_1min
FULL OUTER JOIN agg_hydraulics_1min h
    ON COALESCE(d.bucket_1min, m.bucket_1min) = h.bucket_1min;
""")

✅ SQL ejecutado


## 11. Tratamiento responsable de faltantes

No usamos `COALESCE(sensor, 0)`, porque “sin observación” no equivale a “medición cero”.

Para estado operativo puede aplicarse *forward fill* dentro del mismo día. La partición diaria impide arrastrar un estado de diciembre hasta enero.

In [30]:
run_sql("""
CREATE OR REPLACE VIEW machine_1min_filled_state AS
SELECT
    * EXCLUDE (status_process, status_mode),
    last_value(status_process IGNORE NULLS) OVER (
        PARTITION BY CAST(bucket_1min AS DATE)
        ORDER BY bucket_1min
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS status_process,
    last_value(status_mode IGNORE NULLS) OVER (
        PARTITION BY CAST(bucket_1min AS DATE)
        ORDER BY bucket_1min
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS status_mode
FROM machine_1min_base;
""")

✅ SQL ejecutado


## 12. Feature engineering con SQL

Crearemos:

- Indicador de operación.
- Diferencia de corriente respecto al minuto anterior.
- Diferencia de presión.
- Relación carga hidráulica/corriente.
- Relación torque/velocidad.
- Media y desviación móvil de cinco observaciones.
- Z-score móvil.

In [31]:
run_sql("""
CREATE OR REPLACE TABLE features_machine_1min AS
WITH lagged AS (
    SELECT
        *,
        LAG(drive_1_current_avg_nonzero) OVER (
            PARTITION BY CAST(bucket_1min AS DATE)
            ORDER BY bucket_1min
        ) AS previous_drive_current,
        LAG(pump_pressure_avg_nonzero) OVER (
            PARTITION BY CAST(bucket_1min AS DATE)
            ORDER BY bucket_1min
        ) AS previous_pump_pressure,
        AVG(drive_1_current_avg_nonzero) OVER (
            PARTITION BY CAST(bucket_1min AS DATE)
            ORDER BY bucket_1min
            ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
        ) AS drive_current_ma_5,
        STDDEV_POP(drive_1_current_avg_nonzero) OVER (
            PARTITION BY CAST(bucket_1min AS DATE)
            ORDER BY bucket_1min
            ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
        ) AS drive_current_std_5
    FROM machine_1min_filled_state
)
SELECT
    *,
    CASE
        WHEN status_process IS NOT NULL
         AND status_process <> 'No Operating Mode Selected'
        THEN 1 ELSE 0
    END AS is_operating,
    drive_1_current_avg_nonzero - previous_drive_current
        AS drive_current_delta_1min,
    pump_pressure_avg_nonzero - previous_pump_pressure
        AS pump_pressure_delta_1min,
    pump_pressure_avg_nonzero
        / NULLIF(ABS(drive_1_current_avg_nonzero), 0)
        AS hydraulic_load_per_amp,
    ABS(torque_avg_nonzero)
        / NULLIF(chuck_speed_avg_nonzero, 0)
        AS torque_speed_ratio,
    (drive_1_current_avg_nonzero - drive_current_ma_5)
        / NULLIF(drive_current_std_5, 0)
        AS drive_current_rolling_zscore
FROM lagged;
""")

✅ SQL ejecutado


In [32]:
show_sql("""
SELECT
    bucket_1min,
    status_process,
    status_mode,
    is_operating,
    drive_1_current_avg_nonzero,
    drive_current_delta_1min,
    pump_pressure_avg_nonzero,
    hydraulic_load_per_amp,
    torque_speed_ratio,
    drive_current_rolling_zscore
FROM features_machine_1min
WHERE is_operating = 1
ORDER BY bucket_1min
LIMIT 30
""", max_rows=30)

,bucket_1min,status_process,status_mode,is_operating,drive_1_current_avg_nonzero,drive_current_delta_1min,pump_pressure_avg_nonzero,hydraulic_load_per_amp,torque_speed_ratio,drive_current_rolling_zscore
0,2025-12-11 10:16:00,Pulling Mode Selected,MU,1,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-12-11 10:17:00,Pulling Mode Selected,MU,1,-11.000000,NaN,NaN,NaN,NaN,NaN
2,2025-12-11 10:27:00,Piloting Mode Selected,RF,1,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-12-11 10:28:00,Piloting Mode Selected,RF,1,0.500000,NaN,NaN,NaN,NaN,1.000000
4,2025-12-11 10:31:00,Pulling Mode Selected,MU,1,-18.058824,-18.558824,NaN,NaN,1.929825,-1.116445
5,2025-12-11 10:32:00,Piloting Mode Selected,RF,1,-18.000000,0.058824,NaN,NaN,3.047619,-0.839794
6,2025-12-11 10:33:00,Piloting Mode Selected,RF,1,-17.600000,0.400000,NaN,NaN,0.668896,-0.541260
7,2025-12-11 10:34:00,Piloting Mode Selected,RF,1,8.333333,25.933333,NaN,NaN,NaN,1.543874
8,2025-12-11 10:35:00,Piloting Mode Selected,RF,1,-21.800000,-30.133333,NaN,NaN,5.742331,-0.762364
9,2025-12-11 10:36:00,Piloting Mode Selected,RR,1,22.500000,44.300000,NaN,NaN,3.750000,1.582995


Filas devueltas: 30


,bucket_1min,status_process,status_mode,is_operating,drive_1_current_avg_nonzero,drive_current_delta_1min,pump_pressure_avg_nonzero,hydraulic_load_per_amp,torque_speed_ratio,drive_current_rolling_zscore
0,2025-12-11 10:16:00,Pulling Mode Selected,MU,1,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-12-11 10:17:00,Pulling Mode Selected,MU,1,-11.000000,NaN,NaN,NaN,NaN,NaN
2,2025-12-11 10:27:00,Piloting Mode Selected,RF,1,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-12-11 10:28:00,Piloting Mode Selected,RF,1,0.500000,NaN,NaN,NaN,NaN,1.000000
4,2025-12-11 10:31:00,Pulling Mode Selected,MU,1,-18.058824,-18.558824,NaN,NaN,1.929825,-1.116445
5,2025-12-11 10:32:00,Piloting Mode Selected,RF,1,-18.000000,0.058824,NaN,NaN,3.047619,-0.839794
6,2025-12-11 10:33:00,Piloting Mode Selected,RF,1,-17.600000,0.400000,NaN,NaN,0.668896,-0.541260
7,2025-12-11 10:34:00,Piloting Mode Selected,RF,1,8.333333,25.933333,NaN,NaN,NaN,1.543874
8,2025-12-11 10:35:00,Piloting Mode Selected,RF,1,-21.800000,-30.133333,NaN,NaN,5.742331,-0.762364
9,2025-12-11 10:36:00,Piloting Mode Selected,RR,1,22.500000,44.300000,NaN,NaN,3.750000,1.582995


### Ejercicio 5 — Desviaciones estadísticas

Encuentra minutos donde:

- `is_operating = 1`
- `ABS(drive_current_rolling_zscore) >= 2`

Esto identifica desviaciones locales, no fallas confirmadas.

In [33]:
# Solución
show_sql("""
SELECT
    bucket_1min,
    status_process,
    status_mode,
    drive_1_current_avg_nonzero,
    drive_current_ma_5,
    drive_current_std_5,
    drive_current_rolling_zscore
FROM features_machine_1min
WHERE is_operating = 1
  AND ABS(drive_current_rolling_zscore) >= 2
ORDER BY ABS(drive_current_rolling_zscore) DESC
""")

,bucket_1min,status_process,status_mode,drive_1_current_avg_nonzero,drive_current_ma_5,drive_current_std_5,drive_current_rolling_zscore


Filas devueltas: 0


,bucket_1min,status_process,status_mode,drive_1_current_avg_nonzero,drive_current_ma_5,drive_current_std_5,drive_current_rolling_zscore


## 13. Reporte de calidad

Una canalización debe guardar evidencia de sus validaciones, no solo producir una tabla final.

In [34]:
run_sql("""
CREATE OR REPLACE TABLE data_quality_report AS
SELECT
    current_timestamp AS execution_timestamp,
    'clean_drilling' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE timestamp IS NULL) AS invalid_timestamp,
    COUNT(*) FILTER (WHERE thrust_to_head IS NULL) AS null_primary_measurement,
    COUNT(*) FILTER (WHERE penetration = 0) AS zero_auxiliary_measurement,
    COUNT(*) - COUNT(DISTINCT record_id) AS duplicated_record_id
FROM clean_drilling

UNION ALL

SELECT
    current_timestamp,
    'clean_hydraulics',
    COUNT(*),
    COUNT(*) FILTER (WHERE timestamp IS NULL),
    COUNT(*) FILTER (WHERE pressure_a IS NULL),
    COUNT(*) FILTER (WHERE oil_temperature = 0),
    COUNT(*) - COUNT(DISTINCT record_id)
FROM clean_hydraulics

UNION ALL

SELECT
    current_timestamp,
    'clean_motors',
    COUNT(*),
    COUNT(*) FILTER (WHERE timestamp IS NULL),
    COUNT(*) FILTER (WHERE drive_1_current IS NULL),
    COUNT(*) FILTER (WHERE drive_2_current = 0),
    COUNT(*) - COUNT(DISTINCT record_id)
FROM clean_motors;
""")

show_sql("""
SELECT *
FROM data_quality_report
ORDER BY table_name
""")

✅ SQL ejecutado


,execution_timestamp,table_name,total_rows,invalid_timestamp,null_primary_measurement,zero_auxiliary_measurement,duplicated_record_id
0,2026-07-24 23:06:58.833238+00:00,clean_drilling,183,0,0,183,0
1,2026-07-24 23:06:58.833238+00:00,clean_hydraulics,1213,0,0,1213,0
2,2026-07-24 23:06:58.833238+00:00,clean_motors,508,0,0,508,0


Filas devueltas: 3


,execution_timestamp,table_name,total_rows,invalid_timestamp,null_primary_measurement,zero_auxiliary_measurement,duplicated_record_id
0,2026-07-24 23:06:58.833238+00:00,clean_drilling,183,0,0,183,0
1,2026-07-24 23:06:58.833238+00:00,clean_hydraulics,1213,0,0,1213,0
2,2026-07-24 23:06:58.833238+00:00,clean_motors,508,0,0,508,0


### Ejercicio 6 — Regla de aceptación

Crea `quality_status`:

- `FAIL` si existe timestamp inválido.
- `FAIL` si existe `record_id` duplicado.
- `WARNING` si más de 95% de la variable auxiliar está en cero.
- `PASS` en otro caso.

In [35]:
# Solución para motores
show_sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE timestamp IS NULL) AS invalid_timestamp,
    COUNT(*) - COUNT(DISTINCT record_id) AS duplicated_record_id,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE drive_2_current = 0)
        / NULLIF(COUNT(*), 0),
        2
    ) AS drive_2_current_zero_pct,
    CASE
        WHEN COUNT(*) FILTER (WHERE timestamp IS NULL) > 0 THEN 'FAIL'
        WHEN COUNT(*) - COUNT(DISTINCT record_id) > 0 THEN 'FAIL'
        WHEN 1.0 * COUNT(*) FILTER (WHERE drive_2_current = 0)
             / NULLIF(COUNT(*), 0) > 0.95
        THEN 'WARNING'
        ELSE 'PASS'
    END AS quality_status
FROM clean_motors
""")

,total_rows,invalid_timestamp,duplicated_record_id,drive_2_current_zero_pct,quality_status
0,508,0,0,100.0,WARNING


Filas devueltas: 1


,total_rows,invalid_timestamp,duplicated_record_id,drive_2_current_zero_pct,quality_status
0,508,0,0,100.0,WARNING


## 14. Matriz final para análisis

La consulta conserva únicamente ventanas clasificadas como operación. No elimina automáticamente nulos de sensores, porque cada modelo requerirá una estrategia distinta.

In [36]:
final_df = show_sql("""
SELECT
    bucket_1min,
    status_process,
    status_mode,
    is_operating,
    thrust_to_head_avg_nonzero,
    chuck_speed_avg_nonzero,
    torque_avg_nonzero,
    drive_1_current_avg_nonzero,
    voltage_avg_nonzero,
    total_drive_power_avg_nonzero,
    pressure_a_avg_nonzero,
    pump_pressure_avg_nonzero,
    hydraulic_pressure_efficiency_avg_nonzero,
    hydraulic_system_efficiency_avg_nonzero,
    drive_current_delta_1min,
    pump_pressure_delta_1min,
    hydraulic_load_per_amp,
    torque_speed_ratio,
    drive_current_rolling_zscore
FROM features_machine_1min
WHERE is_operating = 1
ORDER BY bucket_1min
""", max_rows=50)

,bucket_1min,status_process,status_mode,is_operating,thrust_to_head_avg_nonzero,chuck_speed_avg_nonzero,torque_avg_nonzero,drive_1_current_avg_nonzero,voltage_avg_nonzero,total_drive_power_avg_nonzero,pressure_a_avg_nonzero,pump_pressure_avg_nonzero,hydraulic_pressure_efficiency_avg_nonzero,hydraulic_system_efficiency_avg_nonzero,drive_current_delta_1min,pump_pressure_delta_1min,hydraulic_load_per_amp,torque_speed_ratio,drive_current_rolling_zscore
0,2025-12-11 10:16:00,Pulling Mode Selected,MU,1,281.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-12-11 10:17:00,Pulling Mode Selected,MU,1,NaN,NaN,NaN,-11.000000,454.250000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-12-11 10:27:00,Piloting Mode Selected,RF,1,281.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-12-11 10:28:00,Piloting Mode Selected,RF,1,NaN,NaN,NaN,0.500000,455.333333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000
4,2025-12-11 10:31:00,Pulling Mode Selected,MU,1,NaN,2.850000,5.500000,-18.058824,456.000000,2.230769,NaN,NaN,NaN,NaN,-18.558824,NaN,NaN,1.929825,-1.116445
5,2025-12-11 10:32:00,Piloting Mode Selected,RF,1,NaN,3.500000,10.666667,-18.000000,453.933333,3.888889,NaN,NaN,NaN,NaN,0.058824,NaN,NaN,3.047619,-0.839794
6,2025-12-11 10:33:00,Piloting Mode Selected,RF,1,NaN,5.980000,4.000000,-17.600000,455.150000,7.533333,NaN,NaN,NaN,NaN,0.400000,NaN,NaN,0.668896,-0.541260
7,2025-12-11 10:34:00,Piloting Mode Selected,RF,1,NaN,NaN,NaN,8.333333,462.000000,NaN,NaN,NaN,NaN,NaN,25.933333,NaN,NaN,NaN,1.543874
8,2025-12-11 10:35:00,Piloting Mode Selected,RF,1,NaN,2.716667,15.600000,-21.800000,454.789474,3.000000,NaN,NaN,NaN,NaN,-30.133333,NaN,NaN,5.742331,-0.762364
9,2025-12-11 10:36:00,Piloting Mode Selected,RR,1,NaN,2.800000,-10.500000,22.500000,455.157895,4.416667,NaN,NaN,NaN,NaN,44.300000,NaN,NaN,3.750000,1.582995


Filas devueltas: 87


## 15. Exportar resultados

Puedes exportar la tabla analítica a Parquet y descargar tanto el archivo como la base DuckDB completa.

In [37]:
PARQUET_PATH = "/content/features_machine_1min.parquet"

con.execute(f"""
COPY (
    SELECT *
    FROM features_machine_1min
    ORDER BY bucket_1min
)
TO '{PARQUET_PATH}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

con.execute("CHECKPOINT")
print("✅ Archivos preparados")

✅ Archivos preparados


In [38]:
from google.colab import files

# Ejecuta una descarga por vez si el navegador bloquea descargas múltiples.
files.download(PARQUET_PATH)
# files.download(DB_PATH)  # Descomenta para descargar toda la base DuckDB.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 16. Conclusiones

### Resultado

- 63 CSV leídos directamente por DuckDB.
- 36,019 registros reconciliados.
- Nueve tablas `raw` y nueve tablas `clean`.
- Tipos convertidos sin detener el proceso ante valores inválidos.
- Duplicados temporales tratados mediante una regla explícita.
- Registros digitales decodificados bit a bit.
- Series alineadas en ventanas de un minuto.
- Features temporales creadas con SQL.
- Reporte de calidad reproducible.
- Base `machine_data.duckdb` reutilizable.

### Para avanzar a mantenimiento predictivo real

Integra una tabla con:

```text
machine_id
component_id
failure_timestamp
failure_type
maintenance_order
corrective_action
downtime_minutes
```

Las features deben calcularse usando exclusivamente información anterior a cada evento para evitar *data leakage*.

### Preguntas de discusión

1. ¿Los ceros significan paro, pérdida de señal o valores reales?
2. ¿Qué unidad tiene cada sensor?
3. ¿Qué representa cada bit del PLC?
4. ¿Un minuto es una ventana adecuada?
5. ¿Qué eventos del CMMS pueden utilizarse como verdad de terreno?

## Referencias

- Lectura de CSV: https://duckdb.org/docs/stable/data/csv/overview
- Cliente Python: https://duckdb.org/docs/stable/clients/python/overview
- Funciones de ventana: https://duckdb.org/docs/stable/sql/functions/window_functions
- Funciones de fecha y tiempo: https://duckdb.org/docs/stable/sql/functions/date